# EDA — ChurnGuard (datos reales)

**Qué es:** análisis exploratorio sobre el dataset real `data/playnova_real.db` (16.447 jugadores etiquetados de 18.099 reseñas públicas de Steam, 6 F2P, sep-2026).

**De dónde sale todo:** cada número y figura de este notebook sale de la base de datos. Las figuras se guardan en `docs/img/eda_*.png` y alimentan la web (`web/`) y `docs/04_analisis.md`.

**Cómo usarlo:** abrir en Jupyter (`jupyter notebook notebooks/EDA.ipynb`) y ejecutar de arriba a abajo. No necesita Streamlit.

> Nota honesta: las tasas por título son tasas *dentro de la muestra de reseñistas*, no la retención global de esos juegos. Lo que generaliza son las relaciones (validadas juego por juego en `src/eval_per_title.py`).

In [ ]:
import json
import os
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("dark_background")

CYAN, VIOLET, RED, AMBER = "#22d3ee", "#8b5cf6", "#ef4444", "#f59e0b"
NAMES = {570: "Dota 2", 440: "TF2", 230410: "Warframe", 238960: "PoE",
         1172470: "Apex", 1097150: "Fall Guys"}
ORDER = ["Dota 2", "Apex", "TF2", "Warframe", "PoE", "Fall Guys"]

# Rutas robustas: funciona ejecutado desde la raíz o desde notebooks/
DB_CANDIDATES = ["data/playnova_real.db", "../data/playnova_real.db"]
IMG_CANDIDATES = ["docs/img", "../docs/img"]
METRICS_CANDIDATES = ["models/real_metrics.json", "../models/real_metrics.json"]

DB = next(p for p in DB_CANDIDATES if os.path.exists(p))
IMG = next(p for p in IMG_CANDIDATES if os.path.exists(p))
METRICS = next(p for p in METRICS_CANDIDATES if os.path.exists(p))
print("DB:", DB, "| IMG:", IMG, "| METRICS:", METRICS)


def style_ax(ax, title, xlabel="", ylabel=""):
    ax.set_title(title, color=CYAN, fontsize=12, pad=10)
    ax.set_xlabel(xlabel, color="#a0a0c0")
    ax.set_ylabel(ylabel, color="#a0a0c0")
    ax.tick_params(colors="#a0a0c0")

## 0 · Carga y resumen global

In [ ]:
conn = sqlite3.connect(DB)
df = pd.read_sql("SELECT * FROM reviews WHERE churn IS NOT NULL", conn)
conn.close()
df["title"] = df["appid"].map(NAMES)

print(f"Filas: {len(df)} · churn={df['churn'].mean():.1%} · "
      f"high_value={df['high_value'].mean():.1%} · recomienda={df['voted_up'].mean():.1%}")
df.groupby("title").agg(n=("churn", "size"), churn=("churn", "mean"),
    high_value=("high_value", "mean"),
    med_horas_reseña=("hours_at_review", "median")).round({"churn": 3, "high_value": 3, "med_horas_reseña": 1})

## 1 · Estado del jugador por título

Juegos vivos vs juegos fríos: dónde priorizar Live-Ops. Churn (inactivos hoy) vs high-value (proxy top-25% horas + recomienda).

In [ ]:
t = df.groupby("title").agg(churn=("churn", "mean"), high_value=("high_value", "mean")).loc[ORDER]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(t))
ax.bar(x - 0.2, t["churn"] * 100, 0.4, label="Churn (inactivos hoy)", color=RED)
ax.bar(x + 0.2, t["high_value"] * 100, 0.4, label="High-value (proxy)", color=CYAN)
ax.set_xticks(x, t.index)
style_ax(ax, "1 · Estado del jugador por título (muestra de reseñistas, sep-2026)", ylabel="% de reseñistas")
leg = ax.legend()
leg.get_frame().set_facecolor("#1e1b3a")
plt.tight_layout()
plt.savefig(f"{IMG}/eda_1_estado_titulo.png", bbox_inches="tight")
plt.show()

## 2 · El engagement temprano decide

Churn por cuartil de horas jugadas en el momento de la reseña. Q1 (pocas horas) vs Q4 (muchas horas).

In [ ]:
d = df.copy()
d["q"] = pd.qcut(d["hours_at_review"], 4, labels=["Q1\n(pocas h.)", "Q2", "Q3", "Q4\n(muchas h.)"])
t2 = d.groupby("q", observed=True)["churn"].mean()

fig, ax = plt.subplots(figsize=(8, 5))
t2.plot(kind="bar", ax=ax, color=VIOLET, edgecolor=CYAN)
for i, v in enumerate(t2.values):
    ax.text(i, v + 0.008, f"{v:.1%}", ha="center", color="#e8e8f0", fontsize=10)
style_ax(ax, "2 · Churn según horas jugadas en el momento de la reseña",
         xlabel="Cuartil de engagement temprano", ylabel="Tasa de churn")
plt.tight_layout()
plt.savefig(f"{IMG}/eda_2_engagement.png", bbox_inches="tight")
plt.show()
t2

## 3 · Satisfacción ≠ retención

Recomendar el juego apenas mueve la aguja del churn. Gustar no retiene; retiene el hábito temprano.

In [ ]:
t3 = df.groupby("voted_up")["churn"].mean()

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(["No recomienda", "Recomienda"], t3.values * 100, color=[AMBER, CYAN])
for i, v in enumerate(t3.values * 100):
    ax.text(i, v + 0.5, f"{v:.1f}%", ha="center", color="#e8e8f0")
style_ax(ax, "3 · Satisfacción ≠ retención", ylabel="Tasa de churn")
plt.tight_layout()
plt.savefig(f"{IMG}/eda_3_voto.png", bbox_inches="tight")
plt.show()
t3

## 4 · Las primeras horas separan (distribución)

Distribución de horas tempranas (escala log1p): churned vs activos, con sus medianas.

In [ ]:
med_ch = df.loc[df["churn"] == 1, "hours_at_review"].median()
med_ac = df.loc[df["churn"] == 0, "hours_at_review"].median()
print(f"Mediana horas en reseña: churned {med_ch:.1f} h vs activos {med_ac:.1f} h")

fig, ax = plt.subplots(figsize=(8, 5))
ch = np.log1p(df.loc[df["churn"] == 1, "hours_at_review"])
ac = np.log1p(df.loc[df["churn"] == 0, "hours_at_review"])
ax.hist(ch, bins=40, alpha=0.6, label="Churned", color=RED)
ax.hist(ac, bins=40, alpha=0.5, label="Activos", color=CYAN)
ax.axvline(np.log1p(med_ch), color=RED, linestyle="--", label="Mediana churned")
ax.axvline(np.log1p(med_ac), color=CYAN, linestyle="--", label="Mediana activos")
style_ax(ax, "4 · Las primeras horas separan: distribución (log1p)",
         xlabel="log(1 + horas en reseña)", ylabel="N.º jugadores")
leg = ax.legend()
leg.get_frame().set_facecolor("#1e1b3a")
plt.tight_layout()
plt.savefig(f"{IMG}/eda_4_dist_horas.png", bbox_inches="tight")
plt.show()

## 5 · La antigüedad de la señal pesa

Churn por cuartil de `review_age_days`. Es ritmo de reseñas = fase del título (limitación documentada: ventana desigual entre juegos).

In [ ]:
d = df.copy()
d["q_age"] = pd.qcut(d["review_age_days"], 4, labels=["Reciente", "Q2", "Q3", "Antigua"])
t5 = d.groupby("q_age", observed=True)["churn"].mean()

fig, ax = plt.subplots(figsize=(8, 5))
t5.plot(kind="bar", ax=ax, color=AMBER, edgecolor=CYAN)
for i, v in enumerate(t5.values):
    ax.text(i, v + 0.008, f"{v:.1%}", ha="center", color="#e8e8f0", fontsize=10)
style_ax(ax, "5 · Churn vs antigüedad de la señal (review_age_days)",
         xlabel="Cuartil de antigüedad", ylabel="Tasa de churn")
plt.tight_layout()
plt.savefig(f"{IMG}/eda_5_antiguedad.png", bbox_inches="tight")
plt.show()
t5

## 6 · High-value estable por título

Proxy transparente (top-25% horas + recomienda): estable entre 15–24% por título.

In [ ]:
t6 = df.groupby("title").agg(hv=("high_value", "mean")).loc[ORDER]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(t6.index, t6["hv"] * 100, color=CYAN, edgecolor=VIOLET)
ax.axhline(df["high_value"].mean() * 100, color=AMBER, linestyle="--",
           label=f"Global {df['high_value'].mean():.1%}")
for i, v in enumerate(t6["hv"].values * 100):
    ax.text(i, v + 0.4, f"{v:.1f}%", ha="center", color="#e8e8f0", fontsize=9)
style_ax(ax, "6 · High-value estable por título (proxy top-25% horas + recomienda)",
         ylabel="% high-value")
leg = ax.legend()
leg.get_frame().set_facecolor("#1e1b3a")
ax.tick_params(axis="x", rotation=12)
plt.tight_layout()
plt.savefig(f"{IMG}/eda_6_highvalue.png", bbox_inches="tight")
plt.show()
t6

## 7 · Qué pesa en los modelos

Importancias reales de los 2 modelos (`src/train_real.py`, `models/real_metrics.json`).

In [ ]:
with open(METRICS, encoding="utf-8") as f:
    m = json.load(f)
ch = pd.Series(m["churn_model"]["top_features"]).sort_values()
cv = pd.Series(m["conversion_model"]["top_features"]).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ch.plot(kind="barh", ax=axes[0], color=VIOLET, edgecolor=CYAN)
axes[0].set_title("7a · Churn: qué pesa (importancias RF)", color=CYAN, fontsize=11)
cv.plot(kind="barh", ax=axes[1], color=CYAN, edgecolor=VIOLET)
axes[1].set_title("7b · High-value: qué pesa (importancias RF)", color=CYAN, fontsize=11)
for a in axes:
    a.tick_params(colors="#a0a0c0")
plt.tight_layout()
plt.savefig(f"{IMG}/eda_7_importancias.png", bbox_inches="tight")
plt.show()
print("churn ROC:", m["churn_model"].get("roc_auc"), "| F1:", m["churn_model"].get("f1"))
print("conversion ROC:", m["conversion_model"].get("roc_auc"), "| F1:", m["conversion_model"].get("f1"))

## Descubrimientos (números de este EDA)

- **D1 · Engagement temprano decide:** Q1 → Q4 cae del ~35% al ~13%. Mediana churned ~37 h vs activos ~121 h.
- **D2 · Cada título vive su fase:** Dota 2 / Apex (vivos) vs PoE / Fall Guys (fríos). Tasas en muestra de reseñistas, no retención global.
- **D3 · Satisfacción ≠ retención:** recomienda ~26,5% vs no recomienda ~29,6%. Gustar no retiene; retiene el hábito.
- **D4 · High-value estable:** ~20% global, 15–24% por título (proxy top-25% horas + recomienda).
- **D5 · Antigüedad pesa:** es ritmo de reseñas = fase del título (ventana desigual, documentado como limitación).
- **D6 · Modelos:** churn ROC 0.930 / F1 0.817; conversión ROC 0.972 / F1 0.847; validez por título churn ROC 0.891–0.924 (`src/eval_per_title.py`).

Figuras guardadas en `docs/img/eda_*.png`. Detalle narrado en `docs/04_analisis.md`.